In [1]:
import math
import time
import torch
import numpy as np
import torch.nn.functional as F
import matplotlib.pyplot as plt

from utils.metrics import PIVmetric
from utils.visualizer import plot_flow_field
from CMax import ContrastMaximizationLoss

# 数据加载：事件、光流
from generator.flow import FlowManager
from generator.images import config_events
from generator.loader import EventManager
from utils.metrics import PIVmetric

# 检查CUDA是否可用
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

CUDA acceleration: OK
cuda


In [2]:
max_v = 5

flow0 = FlowManager(mode="data", data_path="./data/datasets/JHTDB_isotropic1024_hd_00001_flow.npz")
flow1 = FlowManager(mode="data", data_path="./data/datasets/DNS_turbulence_00001_flow.npz")
flow2 = FlowManager(mode="data", data_path="./data/datasets/SQG_00001_flow.npz")

flow3 = FlowManager(mode="analytical", name="cellular_flow", vmax=max_v, p=256)
flow4 = FlowManager(mode="analytical", name="lamb_oseen", x_c=128, y_c=128, Gamma=3e3, rc=60)
flow5 = FlowManager(mode="analytical", name="shear_flow", vmax=max_v, p=256)

flow6 = FlowManager(mode="data", data_path="./data/datasets/backstep_Re1500_00001_flow.npz")

flow7 = FlowManager(mode="data", data_path="./data/datasets/JHTDB_mhd1024_hd_00001_flow.npz")
flow8 = FlowManager(mode="data", data_path="./data/datasets/JHTDB_channel_hd_00001_flow.npz")

flow9 = FlowManager(mode="analytical", name="gs_flow", max_v=5.0, n_vortex=20, seed=1)
flow10 = FlowManager(mode="analytical", name="solid_rot", x_c=128, y_c=128, omega=0.005525*max_v)
flow11 = FlowManager(mode="analytical", name="uniform", uc=max_v/np.sqrt(2), vc=max_v/np.sqrt(2))
flow12 = FlowManager(mode="analytical", name="sin_flow", x_c=128,y_c=128, a=16, b=128, scale=max_v)
flow13 = FlowManager(mode="data", data_path="./data/datasets/cylinder_Re150_00001_flow.npz")

In [5]:
from gaussian_flow import GaussianFlowConfig, run_gaussian_flow, train_gaussian_flow, render_gaussian_flow
# from profile_cuda_train import profile_train_timing_cuda

for flow_index in range(1):
    
    flow_name = f"flow{flow_index}"
    flow_func = globals()[flow_name].evaluate

    cfg_evts = config_events()
    cfg_evts.time_length = 5
    cfg_evts.d = 0.5
    cfg_evts.density = 0.05
    
    event_loader = EventManager(source_type="simulator", path=None, cfg=cfg_evts, flow_func=flow_func)

    evts, (u, v) = event_loader.load()
    flowgt = torch.stack([u, v], dim=0).to(device)

    events = evts[(evts[:, 2] > 0) & (evts[:, 2] < 3)]

    h_gt = u.cpu().numpy()
    w_gt = v.cpu().numpy()

    # cfg = GaussianFlowConfig(verbose=True, init_n=200, enable_jacobian=True, phase=[50, 100, 125], tile_size = 32, 
    #                          iterations=150, density_interval=25, cmax_sigma= 0.6, cmax_kernel_size = 5,
    #                         cmax_loss_type= 'l2_max', w_div = 5, density_alpha = 0, w_smooth = 0.0)

    cfg = GaussianFlowConfig(verbose=False, iterations=300, init_n=300, enable_jacobian=True,
                             w_div = 10, w_smooth = 0, cmax_loss_type= 'l2_max', phase=[50, 200, 250],
                             cmax_sigma= 0.6, cmax_kernel_size = 5)

    torch.cuda.synchronize()
    start_time = time.time()
    res = run_gaussian_flow(events=events, cfg=cfg, flowgt=None)
    torch.cuda.synchronize()
    end_time = time.time()
    print(f"总时间：{end_time - start_time:.4f}")

    # report = profile_train_timing_cuda(events=events, cfg=cfg, flowgt=None, warmup=10, profile_steps=100, include_density_control=True, print_every=10)

    pred_h, pred_w = res.flow[0], res.flow[1]
    PIVmetric(pred_h, pred_w, h_gt, w_gt, show=True)[:2]

    torch.cuda.empty_cache()

总时间：1.8252
0.1483(RMSE)	0.1204(AEE)	0.0751(AAE,rad)	0.9287(SSIM)	0.9935(MoC)	0(epe1)	0(epe2)	0(epe3)	


# a

In [ ]:
import math
import numpy as np
import torch
import torch.nn.functional as F
import matplotlib.pyplot as plt


# =========================================================
# 1) flow 格式统一
#    默认:
#    flow[0] = v_h
#    flow[1] = v_w
# =========================================================
def ensure_flow_chw(flow):
    flow = torch.as_tensor(flow)

    if flow.ndim == 4:
        if flow.shape[0] != 1:
            raise ValueError(f"Only single-flow batch is supported, got shape {tuple(flow.shape)}")
        flow = flow[0]

    if flow.ndim != 3:
        raise ValueError(f"flow must have 3 dims after squeeze, got {tuple(flow.shape)}")

    if flow.shape[0] == 2:
        out = flow
    elif flow.shape[-1] == 2:
        out = flow.permute(2, 0, 1)
    else:
        raise ValueError(f"Cannot infer flow layout from shape {tuple(flow.shape)}")

    return out.float()


# =========================================================
# 2) 在事件位置采样流场
# =========================================================
def sample_flow_at_events(flow_chw, h, w):
    """
    flow_chw: [2, H, W]
    h, w: [N]
    return:
        vh, vw: [N]
    """
    H, W = flow_chw.shape[1:]
    if H < 2 or W < 2:
        raise ValueError("H and W must be >= 2 for grid_sample.")

    grid_x = 2.0 * (w / (W - 1)) - 1.0
    grid_y = 2.0 * (h / (H - 1)) - 1.0
    grid = torch.stack([grid_x, grid_y], dim=-1).view(1, -1, 1, 2)  # [1, N, 1, 2]

    sampled = F.grid_sample(
        flow_chw.unsqueeze(0),   # [1, 2, H, W]
        grid,
        mode="bilinear",
        padding_mode="border",
        align_corners=True,
    )  # [1, 2, N, 1]

    sampled = sampled.squeeze(0).squeeze(-1)  # [2, N]
    vh = sampled[0]
    vw = sampled[1]
    return vh, vw


# =========================================================
# 3) 双线性 splat 构建 IWE
# =========================================================
def bilinear_splat(h, w, H, W, weights=None):
    """
    h, w: [N] 浮点坐标
    return: [H, W]
    """
    if weights is None:
        weights = torch.ones_like(h, dtype=torch.float32)
    weights = weights.float()

    h0f = torch.floor(h)
    w0f = torch.floor(w)
    dh = h - h0f
    dw = w - w0f

    h0 = h0f.long()
    w0 = w0f.long()
    h1 = h0 + 1
    w1 = w0 + 1

    img = torch.zeros(H * W, device=h.device, dtype=weights.dtype)

    def add_at(h_idx, w_idx, val):
        mask = (
            (h_idx >= 0) & (h_idx < H) &
            (w_idx >= 0) & (w_idx < W) &
            torch.isfinite(val)
        )
        if mask.any():
            flat_idx = h_idx[mask] * W + w_idx[mask]
            img.scatter_add_(0, flat_idx, val[mask])

    add_at(h0, w0, weights * (1 - dh) * (1 - dw))
    add_at(h0, w1, weights * (1 - dh) * dw)
    add_at(h1, w0, weights * dh * (1 - dw))
    add_at(h1, w1, weights * dh * dw)

    return img.view(H, W)


# =========================================================
# 4) 局部窗口核
# =========================================================
def make_window_kernel(window_size, kind="box", sigma=None, device="cpu", dtype=torch.float32, normalize=True):
    if window_size % 2 == 0:
        raise ValueError("window_size must be odd.")

    if kind == "box":
        k = torch.ones((window_size, window_size), device=device, dtype=dtype)
    elif kind == "gaussian":
        if sigma is None:
            sigma = window_size / 6.0
        ax = torch.arange(window_size, device=device, dtype=dtype) - (window_size - 1) / 2
        yy, xx = torch.meshgrid(ax, ax, indexing="ij")
        k = torch.exp(-(yy**2 + xx**2) / (2 * sigma**2))
    else:
        raise ValueError("kind must be 'box' or 'gaussian'")

    if normalize:
        k = k / k.sum()

    return k.view(1, 1, window_size, window_size)


# =========================================================
# 5) 构建正负 IWE
# =========================================================
def build_iwe_pos_neg(flow, events, t_ref=None):
    """
    flow: [2, H, W] or [H, W, 2]
    events: [N, 4] = [h, w, t, p]
    """
    flow = ensure_flow_chw(flow)
    device = flow.device
    dtype = flow.dtype

    events = torch.as_tensor(events, device=device, dtype=dtype)

    H, W = flow.shape[1:]
    h = events[:, 0]
    w = events[:, 1]
    t = events[:, 2]
    p = events[:, 3]

    if t_ref is None:
        t_ref = t.max()

    vh, vw = sample_flow_at_events(flow, h, w)

    h_warp = h - vh * (t - t_ref)
    w_warp = w - vw * (t - t_ref)

    pos_mask = p > 0
    neg_mask = ~pos_mask

    iwe_pos = bilinear_splat(h_warp[pos_mask], w_warp[pos_mask], H, W)
    iwe_neg = bilinear_splat(h_warp[neg_mask], w_warp[neg_mask], H, W)

    return iwe_pos, iwe_neg, h_warp, w_warp


# =========================================================
# 6) RMSE 图
# =========================================================
def flow_rmse_map(pred_flow, gt_flow):
    pred = ensure_flow_chw(pred_flow)
    gt = ensure_flow_chw(gt_flow).to(pred.device, pred.dtype)

    if pred.shape != gt.shape:
        raise ValueError(f"pred and gt must have same shape, got {tuple(pred.shape)} vs {tuple(gt.shape)}")

    rmse = torch.sqrt(((pred - gt) ** 2).mean(dim=0))
    return rmse


# =========================================================
# 7) 局部方差 / 局部和
# =========================================================
def local_variance_divide_sum_map(iwe, window_size=11, kind="box", sigma=None, eps=1e-8):
    """
    返回:
        var_map       : 局部方差图
        local_sum_map : 局部窗口内 IWE 和
        score_map     : var_map / (local_sum_map + eps)
    """
    x = iwe[None, None].float()  # [1,1,H,W]

    mean_kernel = make_window_kernel(
        window_size=window_size,
        kind=kind,
        sigma=sigma,
        device=iwe.device,
        dtype=iwe.dtype,
        normalize=True,
    )

    sum_kernel = make_window_kernel(
        window_size=window_size,
        kind=kind,
        sigma=sigma,
        device=iwe.device,
        dtype=iwe.dtype,
        normalize=False,
    )

    pad = window_size // 2
    x_pad = F.pad(x, (pad, pad, pad, pad), mode="reflect")

    # 局部均值
    mu = F.conv2d(x_pad, mean_kernel)

    # 局部方差: Blur((I - mu)^2)
    resid2 = (x - mu) ** 2
    resid2_pad = F.pad(resid2, (pad, pad, pad, pad), mode="reflect")
    var_map = F.conv2d(resid2_pad, mean_kernel)

    # 局部和
    local_sum_map = F.conv2d(x_pad, sum_kernel).clamp_min(0.0)

    # 当前使用的局部方差指标
    score_map = var_map / (local_sum_map + eps)

    return var_map[0, 0], local_sum_map[0, 0], score_map[0, 0]


# =========================================================
# 8) 分块平均
# =========================================================
def block_reduce_mean(x, block_h=8, block_w=6):
    """
    x: [H, W]
    return:
        block_map: [nH, nW]
    """
    x = x.float()
    H, W = x.shape
    nH = math.ceil(H / block_h)
    nW = math.ceil(W / block_w)

    block_map = torch.zeros((nH, nW), device=x.device, dtype=x.dtype)

    for bi in range(nH):
        hs = bi * block_h
        he = min((bi + 1) * block_h, H)
        for bj in range(nW):
            ws = bj * block_w
            we = min((bj + 1) * block_w, W)
            block_map[bi, bj] = x[hs:he, ws:we].mean()

    return block_map


def expand_block_map(block_map, H, W, block_h=8, block_w=6):
    """
    把块图恢复到原图大小，便于显示
    """
    block_map = block_map.float()
    nH, nW = block_map.shape
    out = torch.zeros((H, W), device=block_map.device, dtype=block_map.dtype)

    for bi in range(nH):
        hs = bi * block_h
        he = min((bi + 1) * block_h, H)
        for bj in range(nW):
            ws = bj * block_w
            we = min((bj + 1) * block_w, W)
            out[hs:he, ws:we] = block_map[bi, bj]

    return out


# =========================================================
# 9) 取最大/最小比例块
# =========================================================
def ratio_block_mask(block_map, ratio=0.2, largest=True, valid_mask=None):
    """
    block_map: [nH, nW]
    ratio: 比例，例如 0.2 表示 20%
    largest=True  -> 最大20%
    largest=False -> 最小20%
    valid_mask: 可选，只在有效块中选
    """
    flat = block_map.reshape(-1)

    if valid_mask is None:
        valid_flat = torch.ones_like(flat, dtype=torch.bool)
    else:
        valid_flat = valid_mask.reshape(-1).bool()

    valid_idx = torch.where(valid_flat)[0]
    if valid_idx.numel() == 0:
        return torch.zeros_like(block_map, dtype=torch.bool)

    vals = flat[valid_idx]
    K = max(1, int(math.ceil(vals.numel() * ratio)))

    if largest:
        sel_local = torch.topk(vals, k=K, largest=True).indices
    else:
        sel_local = torch.topk(vals, k=K, largest=False).indices

    sel_global = valid_idx[sel_local]

    out = torch.zeros_like(flat, dtype=torch.bool)
    out[sel_global] = True
    return out.view_as(block_map)


# =========================================================
# 10) 可视化主函数
# =========================================================
def visualize_rmse_localscore_blocks(
    flow,
    events,
    flowgt,
    window_size=11,
    window_kind="box",       # "box" or "gaussian"
    window_sigma=None,
    t_ref=None,
    robust_percentile=99.5,  # 原图显示上限分位数
    block_h=8,
    block_w=6,
    mark_ratio=0.2,          # 20%
    sum_threshold=1e-4,      # 局部 score 有效块阈值
    figsize=(16, 12),
):
    """
    显示:
      1) 原图: RMSE, pos local score, neg local score
      2) 分块均值图
      3) 块级二值标记图:
         - RMSE 最大20%
         - Pos score 最小20%
         - Neg score 最小20%
    """
    pred = ensure_flow_chw(flow)
    gt = ensure_flow_chw(flowgt).to(pred.device, pred.dtype)

    # -----------------------------
    # 原始图
    # -----------------------------
    iwe_pos, iwe_neg, _, _ = build_iwe_pos_neg(pred, events, t_ref=t_ref)

    var_pos, sum_pos, score_pos = local_variance_divide_sum_map(
        iwe_pos,
        window_size=window_size,
        kind=window_kind,
        sigma=window_sigma,
    )
    var_neg, sum_neg, score_neg = local_variance_divide_sum_map(
        iwe_neg,
        window_size=window_size,
        kind=window_kind,
        sigma=window_sigma,
    )

    rmse_map = flow_rmse_map(pred, gt)

    H, W = rmse_map.shape

    # -----------------------------
    # 分块
    # -----------------------------
    rmse_block = block_reduce_mean(rmse_map, block_h=block_h, block_w=block_w)
    score_pos_block = block_reduce_mean(score_pos, block_h=block_h, block_w=block_w)
    score_neg_block = block_reduce_mean(score_neg, block_h=block_h, block_w=block_w)

    sum_pos_block = block_reduce_mean(sum_pos, block_h=block_h, block_w=block_w)
    sum_neg_block = block_reduce_mean(sum_neg, block_h=block_h, block_w=block_w)

    valid_pos_block = sum_pos_block > sum_threshold
    valid_neg_block = sum_neg_block > sum_threshold

    # -----------------------------
    # 块级二值标记
    # RMSE: 最大20%
    # Score: 最小20%
    # -----------------------------
    rmse_top_block_mask = ratio_block_mask(
        rmse_block, ratio=mark_ratio, largest=True, valid_mask=None
    )
    pos_bottom_block_mask = ratio_block_mask(
        score_pos_block, ratio=mark_ratio, largest=False, valid_mask=valid_pos_block
    )
    neg_bottom_block_mask = ratio_block_mask(
        score_neg_block, ratio=mark_ratio, largest=False, valid_mask=valid_neg_block
    )

    # 展开到原图大小，便于显示
    rmse_block_expand = expand_block_map(rmse_block, H, W, block_h, block_w)
    score_pos_block_expand = expand_block_map(score_pos_block, H, W, block_h, block_w)
    score_neg_block_expand = expand_block_map(score_neg_block, H, W, block_h, block_w)

    rmse_top_expand = expand_block_map(rmse_top_block_mask.float(), H, W, block_h, block_w)
    pos_bottom_expand = expand_block_map(pos_bottom_block_mask.float(), H, W, block_h, block_w)
    neg_bottom_expand = expand_block_map(neg_bottom_block_mask.float(), H, W, block_h, block_w)

    # -----------------------------
    # 打印信息
    # -----------------------------
    print(f"[Block size] ({block_h}, {block_w})")
    print(f"[Mark ratio] {mark_ratio * 100:.1f}%")
    print(f"[Valid block count] pos = {valid_pos_block.sum().item()}, neg = {valid_neg_block.sum().item()}")
    print(f"[Marked block count] RMSE top = {rmse_top_block_mask.sum().item()}, "
          f"Pos bottom = {pos_bottom_block_mask.sum().item()}, "
          f"Neg bottom = {neg_bottom_block_mask.sum().item()}")

    # -----------------------------
    # 可视化
    # 3 x 3:
    # 第1行 原图
    # 第2行 分块均值
    # 第3行 块级标记
    # -----------------------------
    fig, axes = plt.subplots(3, 3, figsize=figsize, constrained_layout=True)
    axes = axes.ravel()

    maps = [
        ("RMSE", rmse_map.detach().cpu(), "raw"),
        ("Pos local score = var / local_sum", score_pos.detach().cpu(), "raw"),
        ("Neg local score = var / local_sum", score_neg.detach().cpu(), "raw"),

        (f"Block mean RMSE ({block_h}x{block_w})", rmse_block_expand.detach().cpu(), "block"),
        (f"Block mean Pos score ({block_h}x{block_w})", score_pos_block_expand.detach().cpu(), "block"),
        (f"Block mean Neg score ({block_h}x{block_w})", score_neg_block_expand.detach().cpu(), "block"),

        (f"Top {int(mark_ratio*100)}% RMSE blocks", rmse_top_expand.detach().cpu(), "mask"),
        (f"Bottom {int(mark_ratio*100)}% Pos score blocks", pos_bottom_expand.detach().cpu(), "mask"),
        (f"Bottom {int(mark_ratio*100)}% Neg score blocks", neg_bottom_expand.detach().cpu(), "mask"),
    ]

    for ax, (title, img_t, mode) in zip(axes, maps):
        img = img_t.numpy()

        if mode in ["raw", "block"]:
            vmax = np.percentile(img, robust_percentile) if robust_percentile is not None else None
            if vmax is not None:
                vmax = max(vmax, 1e-12)
            im = ax.imshow(img, origin="upper", vmin=0.0, vmax=vmax, interpolation="nearest")
        else:
            im = ax.imshow(img, origin="upper", vmin=0.0, vmax=1.0, interpolation="nearest")

        ax.set_title(title)
        ax.set_xlabel("w")
        ax.set_ylabel("h")
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        # 画块边界
        for hline in range(block_h, H, block_h):
            ax.axhline(hline - 0.5, color="white", linewidth=0.3, alpha=0.5)
        for wline in range(block_w, W, block_w):
            ax.axvline(wline - 0.5, color="white", linewidth=0.3, alpha=0.5)

    plt.show()

    return {
        # 原图
        "iwe_pos": iwe_pos,
        "iwe_neg": iwe_neg,
        "local_var_pos": var_pos,
        "local_var_neg": var_neg,
        "local_sum_pos": sum_pos,
        "local_sum_neg": sum_neg,
        "local_score_pos": score_pos,
        "local_score_neg": score_neg,
        "rmse_map": rmse_map,

        # 分块
        "rmse_block": rmse_block,
        "score_pos_block": score_pos_block,
        "score_neg_block": score_neg_block,
        "valid_pos_block": valid_pos_block,
        "valid_neg_block": valid_neg_block,

        # 标记
        "rmse_top_block_mask": rmse_top_block_mask,
        "pos_bottom_block_mask": pos_bottom_block_mask,
        "neg_bottom_block_mask": neg_bottom_block_mask,
    }

out = visualize_rmse_localscore_blocks(
    flow=torch.tensor(res.flow),
    events=events,
    flowgt=flowgt,
    window_size=11,
    window_kind="box",
    block_h=16,
    block_w=16,
    mark_ratio=0.2,
)



In [ ]:
import math
import torch
import matplotlib.pyplot as plt


# =========================
# 1) 基础工具
# =========================
def bilinear_sample_flow(flow, h, w):
    """
    flow: [2, H, W]
    h, w: [N] float
    return: [N, 2]
    """
    device = flow.device
    _, H, W = flow.shape

    h0 = torch.floor(h).long()
    w0 = torch.floor(w).long()
    h1 = h0 + 1
    w1 = w0 + 1

    h0c = h0.clamp(0, H - 1)
    h1c = h1.clamp(0, H - 1)
    w0c = w0.clamp(0, W - 1)
    w1c = w1.clamp(0, W - 1)

    dh = (h - h0.float()).clamp(0, 1)
    dw = (w - w0.float()).clamp(0, 1)

    wa = (1 - dh) * (1 - dw)  # h0,w0
    wb = (1 - dh) * dw        # h0,w1
    wc = dh * (1 - dw)        # h1,w0
    wd = dh * dw              # h1,w1

    f00 = flow[:, h0c, w0c].permute(1, 0)  # [N,2]
    f01 = flow[:, h0c, w1c].permute(1, 0)
    f10 = flow[:, h1c, w0c].permute(1, 0)
    f11 = flow[:, h1c, w1c].permute(1, 0)

    f = wa[:, None] * f00 + wb[:, None] * f01 + wc[:, None] * f10 + wd[:, None] * f11
    return f


def splat_events_bilinear(h, w, H, W, weight=None):
    """
    将事件双线性 splat 到 HxW 图上
    h, w: [N] float
    weight: [N] or None
    return: [H, W]
    """
    device = h.device
    if weight is None:
        weight = torch.ones_like(h)

    h0 = torch.floor(h).long()
    w0 = torch.floor(w).long()
    h1 = h0 + 1
    w1 = w0 + 1

    dh = h - h0.float()
    dw = w - w0.float()

    wa = (1 - dh) * (1 - dw)
    wb = (1 - dh) * dw
    wc = dh * (1 - dw)
    wd = dh * dw

    img = torch.zeros(H * W, device=device, dtype=torch.float32)

    def add_corner(hh, ww, wwgt):
        valid = (hh >= 0) & (hh < H) & (ww >= 0) & (ww < W)
        if valid.any():
            idx = hh[valid] * W + ww[valid]
            img.scatter_add_(0, idx, (wwgt[valid] * weight[valid]).float())

    add_corner(h0, w0, wa)
    add_corner(h0, w1, wb)
    add_corner(h1, w0, wc)
    add_corner(h1, w1, wd)

    return img.view(H, W)


def blockify_2d(x, block_size):
    """
    x: [H, W]
    return:
      blocks: [Hb*Wb, block_size*block_size]
      Hb, Wb
    """
    H, W = x.shape
    Hb = H // block_size
    Wb = W // block_size
    x = x[:Hb * block_size, :Wb * block_size]
    blocks = x.view(Hb, block_size, Wb, block_size).permute(0, 2, 1, 3).reshape(Hb * Wb, -1)
    return blocks, Hb, Wb


def rankdata_torch(x):
    """
    简单 rank，用于 Spearman
    """
    order = torch.argsort(x)
    ranks = torch.empty_like(order, dtype=torch.float32)
    ranks[order] = torch.arange(len(x), device=x.device, dtype=torch.float32)
    return ranks


def corrcoef_torch(x, y, eps=1e-8):
    x = x.float()
    y = y.float()
    x = x - x.mean()
    y = y - y.mean()
    return (x * y).mean() / (x.std(unbiased=False) * y.std(unbiased=False) + eps)


def top_ratio_mask(score_map, valid_map, ratio=0.2):
    """
    score_map: [Hb, Wb], 越大越差
    valid_map: [Hb, Wb] bool
    """
    vals = score_map[valid_map]
    if vals.numel() == 0:
        raise ValueError("No valid blocks.")
    k = max(1, int(math.ceil(ratio * vals.numel())))
    topk_vals = torch.topk(vals, k=k, largest=True).values
    thresh = topk_vals.min()
    mask = (score_map >= thresh) & valid_map
    return mask


def mask_iou_dice(mask1, mask2, eps=1e-8):
    inter = (mask1 & mask2).sum().float()
    union = (mask1 | mask2).sum().float()
    s1 = mask1.sum().float()
    s2 = mask2.sum().float()
    iou = inter / (union + eps)
    dice = 2 * inter / (s1 + s2 + eps)
    return iou.item(), dice.item()


# =========================
# 2) 核心：块级双时窗一致性信号
# =========================
def compute_block_temporal_consistency(
    events,
    flow,
    block_size=16,
    t_ref=None,
    min_events_per_block=20,
    use_polarity=False,
    eps=1e-6,
):
    """
    events: [N,4] -> (h,w,t,p)
    flow:   [2,H,W]
    return:
      score_map: [Hb,Wb] 越大越差
      valid_map: [Hb,Wb]
      aux: dict
    """
    device = flow.device
    events = events.to(device)
    flow = flow.to(device)

    _, H, W = flow.shape

    h = events[:, 0].float()
    w = events[:, 1].float()
    t = events[:, 2].float()
    p = events[:, 3].float()

    if t_ref is None:
        t_ref = t.median()

    t_mid = t.median()
    mask1 = t <= t_mid
    mask2 = t > t_mid

    if mask1.sum() == 0 or mask2.sum() == 0:
        raise ValueError("One temporal half is empty.")

    def warp_and_render(mask):
        hh = h[mask]
        ww = w[mask]
        tt = t[mask]
        pp = p[mask]

        v = bilinear_sample_flow(flow, hh, ww)  # [N,2], [dh,dw]
        dt = (t_ref - tt)

        # 若你的训练里是 h' = h - v*dt / w' = w - v*dt，请改这里
        h_warp = hh + v[:, 0] * dt
        w_warp = ww + v[:, 1] * dt

        if use_polarity:
            weight = torch.sign(pp)
        else:
            weight = torch.ones_like(pp)

        iwe = splat_events_bilinear(h_warp, w_warp, H, W, weight=weight)
        cnt = splat_events_bilinear(h_warp, w_warp, H, W, weight=torch.ones_like(pp))
        return iwe, cnt

    iwe1, cnt1 = warp_and_render(mask1)
    iwe2, cnt2 = warp_and_render(mask2)

    b1, Hb, Wb = blockify_2d(iwe1.abs(), block_size)
    b2, _, _ = blockify_2d(iwe2.abs(), block_size)
    c1, _, _ = blockify_2d(cnt1, block_size)
    c2, _, _ = blockify_2d(cnt2, block_size)

    sum1 = b1.sum(dim=1, keepdim=True)
    sum2 = b2.sum(dim=1, keepdim=True)
    cnt_sum = c1.sum(dim=1) + c2.sum(dim=1)

    b1n = b1 / (sum1 + eps)
    b2n = b2 / (sum2 + eps)

    # 块级一致性误差：越大越差
    score = (b1n - b2n).abs().mean(dim=1)

    valid = cnt_sum >= float(min_events_per_block)

    score_map = score.view(Hb, Wb)
    valid_map = valid.view(Hb, Wb)

    aux = {
        "iwe1": iwe1,
        "iwe2": iwe2,
        "cnt1": cnt1,
        "cnt2": cnt2,
        "t_ref": t_ref,
        "t_mid": t_mid,
    }
    return score_map, valid_map, aux


# =========================
# 3) 真值块级 RMSE
# =========================
def compute_block_rmse(flow, flowgt, block_size=16):
    """
    flow, flowgt: [2,H,W]
    return:
      rmse_map: [Hb,Wb]
      pixel_rmse: [H,W]
    """
    err = ((flow - flowgt) ** 2).mean(dim=0).sqrt()  # 每像素 RMSE
    blocks, Hb, Wb = blockify_2d(err, block_size)
    rmse_map = blocks.mean(dim=1).view(Hb, Wb)
    return rmse_map, err


# =========================
# 4) 评估：top20%、相关性
# =========================
def evaluate_proxy_against_rmse(
    events,
    flow,
    flowgt,
    block_size=16,
    top_ratio=0.2,
    t_ref=None,
    min_events_per_block=20,
    use_polarity=False,
):
    proxy_map, valid_map, aux = compute_block_temporal_consistency(
        events=events,
        flow=flow,
        block_size=block_size,
        t_ref=t_ref,
        min_events_per_block=min_events_per_block,
        use_polarity=use_polarity,
    )

    rmse_map, pixel_rmse = compute_block_rmse(flow, flowgt, block_size=block_size)

    # 只在 valid blocks 上比较
    proxy_vals = proxy_map[valid_map]
    rmse_vals = rmse_map[valid_map]

    pearson = corrcoef_torch(proxy_vals, rmse_vals).item()
    spearman = corrcoef_torch(rankdata_torch(proxy_vals), rankdata_torch(rmse_vals)).item()

    proxy_top = top_ratio_mask(proxy_map, valid_map, ratio=top_ratio)
    rmse_top = top_ratio_mask(rmse_map, valid_map, ratio=top_ratio)

    iou, dice = mask_iou_dice(proxy_top, rmse_top)

    # 二值 mask 的相关性（phi coefficient）
    phi = corrcoef_torch(
        proxy_top[valid_map].float(),
        rmse_top[valid_map].float()
    ).item()

    metrics = {
        "pearson": pearson,
        "spearman": spearman,
        "top_ratio": top_ratio,
        "top_iou": iou,
        "top_dice": dice,
        "top_phi": phi,
        "num_valid_blocks": int(valid_map.sum().item()),
    }

    return {
        "proxy_map": proxy_map,
        "rmse_map": rmse_map,
        "valid_map": valid_map,
        "proxy_top": proxy_top,
        "rmse_top": rmse_top,
        "pixel_rmse": pixel_rmse,
        "metrics": metrics,
        "aux": aux,
    }


# =========================
# 5) 可视化
# =========================
def upsample_block_map(block_map, block_size, H, W, fill_value=float("nan")):
    Hb, Wb = block_map.shape
    full = torch.full((Hb * block_size, Wb * block_size), fill_value, device=block_map.device)
    for i in range(Hb):
        for j in range(Wb):
            full[i*block_size:(i+1)*block_size, j*block_size:(j+1)*block_size] = block_map[i, j]
    return full[:H, :W]


def visualize_proxy_result(result, block_size, H, W, title_prefix=""):
    proxy_map = result["proxy_map"]
    rmse_map = result["rmse_map"]
    valid_map = result["valid_map"]
    proxy_top = result["proxy_top"]
    rmse_top = result["rmse_top"]
    pixel_rmse = result["pixel_rmse"]
    metrics = result["metrics"]

    proxy_full = upsample_block_map(proxy_map, block_size, H, W).detach().cpu()
    rmse_full  = upsample_block_map(rmse_map, block_size, H, W).detach().cpu()
    valid_full = upsample_block_map(valid_map.float(), block_size, H, W).detach().cpu()
    ptop_full  = upsample_block_map(proxy_top.float(), block_size, H, W).detach().cpu()
    rtop_full  = upsample_block_map(rmse_top.float(), block_size, H, W).detach().cpu()
    overlap_full = (ptop_full > 0.5) & (rtop_full > 0.5)

    fig, axes = plt.subplots(2, 3, figsize=(14, 9))

    im0 = axes[0, 0].imshow(proxy_full, cmap="hot")
    axes[0, 0].set_title("Proxy score (block-wise)")
    plt.colorbar(im0, ax=axes[0, 0], fraction=0.046)

    im1 = axes[0, 1].imshow(rmse_full, cmap="hot")
    axes[0, 1].set_title("GT block RMSE")
    plt.colorbar(im1, ax=axes[0, 1], fraction=0.046)

    im2 = axes[0, 2].imshow(pixel_rmse.detach().cpu(), cmap="hot")
    axes[0, 2].set_title("Pixel RMSE")
    plt.colorbar(im2, ax=axes[0, 2], fraction=0.046)

    axes[1, 0].imshow(ptop_full, cmap="gray")
    axes[1, 0].set_title(f"Proxy top {int(metrics['top_ratio']*100)}%")

    axes[1, 1].imshow(rtop_full, cmap="gray")
    axes[1, 1].set_title(f"GT RMSE top {int(metrics['top_ratio']*100)}%")

    overlap_vis = torch.zeros((*overlap_full.shape, 3), dtype=torch.float32)
    overlap_vis[..., 0] = ptop_full.float()  # red
    overlap_vis[..., 1] = rtop_full.float()  # green
    axes[1, 2].imshow(overlap_vis.numpy())
    axes[1, 2].set_title("Overlap (R=Proxy, G=RMSE)")

    for ax in axes.ravel():
        ax.set_xticks([])
        ax.set_yticks([])

    fig.suptitle(
        f"{title_prefix}\n"
        f"Pearson={metrics['pearson']:.4f}, "
        f"Spearman={metrics['spearman']:.4f}, "
        f"IoU={metrics['top_iou']:.4f}, "
        f"Dice={metrics['top_dice']:.4f}, "
        f"Phi={metrics['top_phi']:.4f}",
        fontsize=12
    )
    plt.tight_layout()
    plt.show()

flow = torch.tensor(res.flow, device=flowgt.device, dtype=flowgt.dtype)  # [2,H,W]
events = events.to(flowgt.device)

result = evaluate_proxy_against_rmse(
    events=events,
    flow=flow,
    flowgt=flowgt,
    block_size=16,          # 8 或 16 都可以
    top_ratio=0.2,          # top 20%
    t_ref=None,             # 默认取事件时间中位数；更建议改成和你 CMax 一致的参考时刻
    min_events_per_block=20,
    use_polarity=False,
)

print(result["metrics"])

H, W = flow.shape[-2:]
visualize_proxy_result(result, block_size=16, H=H, W=W, title_prefix="Block Temporal Consistency")

In [ ]:
    pred_flow_50=torch.tensor(res.flow),
    gt_flow=flowgt,

In [ ]:
import os
import numpy as np
import torch
import matplotlib.pyplot as plt


def plot_event_3d(events,
                  save_path,
                  elev=20,
                  azim=-60,
                  point_size=2,
                  alpha=0.8):
    """
    events: (N,4) -> [h, w, t, p]
    save_path: 保存路径 (e.g. "./event_3d.png")
    """

    # ===== 转 numpy =====
    if isinstance(events, torch.Tensor):
        events = events.detach().cpu().numpy()

    h = events[:, 0]
    w = events[:, 1]
    t = events[:, 2]
    p = events[:, 3]

    # ===== 分极性 =====
    mask_pos = p > 0.5
    mask_neg = p < 0.5

    # ===== 创建图 =====
    fig = plt.figure(figsize=(6, 6))
    ax = fig.add_subplot(111, projection='3d')

    # ===== 画点 =====
    ax.scatter(w[mask_pos], h[mask_pos], t[mask_pos],
               c='red', s=point_size, alpha=alpha)
    ax.scatter(w[mask_neg], h[mask_neg], t[mask_neg],
               c='blue', s=point_size, alpha=alpha)

    # ===== 仅保留三轴（论文风）=====
    ax.set_xlabel('W')
    ax.set_ylabel('H')
    ax.set_zlabel('T')

    ax.view_init(elev=elev, azim=azim)

    # 去掉网格 & 背景
    ax.grid(False)
    ax.xaxis.pane.fill = False
    ax.yaxis.pane.fill = False
    ax.zaxis.pane.fill = False

    # ===== 保存 =====
    os.makedirs(os.path.dirname(save_path), exist_ok=True)
    plt.tight_layout()
    plt.savefig(save_path, dpi=300, bbox_inches='tight', pad_inches=0)
    plt.show()

    print(f"Saved to: {save_path}")

plot_event_3d(events, "./event_3d.png")

In [ ]:
plot_flow_field(pred_h, pred_w, output_file=None, colorbar_flag=False, step=1, quiver_step=16, scale=20, figsize=(5, 3))

In [ ]:
def visualize_coherence_hw(model, pred_flow, weight_map, events=None, arrow_step=16, quiver_scale=50):

    H, W = model.H, model.W

    with torch.no_grad():
        flow = pred_flow.detach().cpu()                    # [2, H, W]
        wmap = weight_map.squeeze().detach().cpu()         # [H, W]

        v_h = flow[0]                                      # [H, W] 竖直分量
        v_w = flow[1]                                      # [H, W] 水平分量
        flow_mag = torch.sqrt(v_h**2 + v_w**2)            # [H, W]

        # 归一化用于背景显示
        flow_mag_norm = (flow_mag - flow_mag.min()) / (flow_mag.max() - flow_mag.min() + 1e-6)

        fig, axes = plt.subplots(1, 3, figsize=(16, 5))

        ax = axes[0]
        im = ax.imshow(wmap.numpy(), cmap='hot')
        ax.set_title('Weight Map')
        ax.set_xlabel('w')
        ax.set_ylabel('h')
        plt.colorbar(im, ax=ax, fraction=0.046, pad=0.04)

        ax = axes[1]
        ax.imshow(flow_mag.numpy(), cmap='viridis')

        h_coords = torch.arange(0, H, arrow_step)
        w_coords = torch.arange(0, W, arrow_step)
        hh, ww = torch.meshgrid(h_coords, w_coords, indexing='ij')
        
        u = v_w[hh, ww].numpy()
        v = v_h[hh, ww].numpy()

        ax.quiver(
            ww.numpy(), hh.numpy(),
            u, -v,
            scale=quiver_scale,
            color='red',
            width=0.003
        )
        ax.set_title('Flow Magnitude + Arrows')
        ax.set_xlabel('w')
        ax.set_ylabel('h')

        ax = axes[2]
        ax.imshow(flow_mag_norm.numpy(), cmap='hot')

        mu_np = model.mu.detach().cpu().numpy()   # [N, 2], [h, w]
        op_np = torch.sigmoid(model.opacity_logit).squeeze().detach().cpu().numpy()

        sc = ax.scatter(
            mu_np[:, 1],   # w
            mu_np[:, 0],   # h
            c=op_np,
            cmap='cool',
            s=15,
            alpha=0.8,
            vmin=0,
            vmax=1
        )
        plt.colorbar(sc, ax=ax, fraction=0.046, pad=0.04)
        ax.set_title('Gaussians on Flow')
        ax.set_xlabel('w')
        ax.set_ylabel('h')
        ax.set_xlim(0, W)
        ax.set_ylim(H, 0)

        plt.tight_layout()
        plt.show()

visualize_coherence_hw(res.model, torch.tensor(res.flow), res.weight_map, events=events, arrow_step=16, quiver_scale=50)

In [ ]:
def analyze_model_stats(model):
    """
    全面分析训练后 GaussianVelocityField 模型的各项参数统计信息
    """
    print("="*60)
    print(f"MODEL STATISTICS REPORT (N={model.num_gaussians})")
    print("="*60)
    
    # 将数据移回 CPU 并转为 NumPy
    with torch.no_grad():
        mu = model.mu.detach().cpu().numpy()
        log_scale = model.log_scale.detach().cpu().numpy()
        scales = np.exp(log_scale)
        rotation = model.rotation.detach().cpu().numpy()
        opacity = torch.sigmoid(model.opacity_logit).detach().cpu().numpy()
        velocity = model.local_velocity.detach().cpu().numpy()
        J = model.J_params.detach().cpu().numpy()

    # --- 1. 位置 (Position) ---
    print(f"\n[1. Position (mu)]")
    print(f"  Range X: {mu[:, 1].min():.2f} ~ {mu[:, 1].max():.2f}")
    print(f"  Range Y: {mu[:, 0].min():.2f} ~ {mu[:, 0].max():.2f}")
    # 检查有多少跑出画布外
    out_bound = ((mu[:,0]<0)|(mu[:,0]>model.H)|(mu[:,1]<0)|(mu[:,1]>model.W)).sum()
    print(f"  Out of bounds: {out_bound} / {model.num_gaussians} ({out_bound/model.num_gaussians*100:.1f}%)")

    # --- 2. 尺度 (Scale) ---
    print(f"\n[2. Scale (pixels)]")
    print(f"  Min: {scales.min():.4f} | Max: {scales.max():.4f}")
    print(f"  Mean: {scales.mean():.4f} | Median: {np.median(scales):.4f}")
    
    # 长宽比 (Anisotropy)
    aspect_ratios = np.max(scales, axis=1) / np.min(scales, axis=1)
    print(f"  Aspect Ratio (Long/Short): Mean={aspect_ratios.mean():.2f}, Max={aspect_ratios.max():.2f}, Median={np.median(aspect_ratios):.2f}")
    if aspect_ratios.max() > 10:
        print("  WARNING: 存在极度细长的高斯，可能导致渲染伪影。")

    # --- 3. 不透明度 (Opacity) ---
    print(f"\n[3. Opacity (alpha)]")
    print(f"  Mean: {opacity.mean():.4f}")
    print(f"  < 0.1 (Transparent): {(opacity < 0.1).sum()} count")
    print(f"  > 0.9 (Solid):       {(opacity > 0.9).sum()} count")
    # 理想情况下，opacity 应该是两极分化的（要么很有用，要么没用）

    # --- 4. 速度 (Velocity) ---
    vel_mag = np.linalg.norm(velocity, axis=1)
    print(f"\n[4. Velocity Magnitude]")
    print(f"  Mean: {vel_mag.mean():.4f} | Max: {vel_mag.max():.4f}")
    print(f"  Min:  {vel_mag.min():.4f}")

    # --- 5. 雅可比 (Jacobian) ---
    J_mag = np.linalg.norm(J, axis=1) # Frobenius norm
    print(f"\n[5. Jacobian Norm]")
    print(f"  Mean: {J_mag.mean():.4f} | Max: {J_mag.max():.4f}")

    print("="*60)
    
    # ==========================================
    # 可视化绘图
    # ==========================================
    fig, axes = plt.subplots(2, 3, figsize=(15, 8))
    fig.suptitle(f'Model Parameter Analysis (N={model.num_gaussians})', fontsize=16)

    # Plot 1: 空间分布散点图
    ax = axes[0, 0]
    ax.set_title("Gaussian Centers (Spatial)")
    # 根据 Opacity 设置颜色深浅
    mask = np.max(scales, axis=1) < 1
    mu_m = mu[mask]
    print(mu_m.shape)
    opacity_m = opacity[mask]
    sc = ax.scatter(mu_m[:, 1], mu_m[:, 0], c=opacity_m, cmap='viridis', s=10, alpha=0.7)
    ax.set_xlim(0, model.W)
    ax.set_ylim(0, model.H)
    ax.invert_yaxis() # 图像坐标系 Y 轴向下
    plt.colorbar(sc, ax=ax, label='Opacity')

    # Plot 2: 尺度分布 (直方图)
    ax = axes[0, 1]
    scales_mask = scales.flatten()[scales.flatten() < 2000]
    ax.set_title("Scale Distribution (Pixels)")
    ax.hist(scales_mask, bins=30, color='skyblue', edgecolor='black')
    ax.set_xlabel("Scale Value")

    # Plot 3: 长宽比分布
    ax = axes[0, 2]
    aspect_ratios = aspect_ratios[aspect_ratios < 10]
    ax.set_title("Aspect Ratio (Anisotropy)")
    ax.hist(aspect_ratios, bins=30, color='salmon', edgecolor='black')
    ax.set_xlabel("Long / Short Axis")

    # Plot 4: 不透明度分布
    ax = axes[1, 0]
    ax.set_title("Opacity Distribution")
    ax.hist(opacity.flatten(), bins=20, range=(0,1), color='lightgreen', edgecolor='black')
    ax.set_xlabel("Alpha (0~1)")
    ax.set_yscale('log') # 使用对数坐标，因为通常 0 附近的数量极大

    # Plot 5: 速度幅度分布
    ax = axes[1, 1]
    ax.set_title("Local Velocity Magnitude")
    ax.hist(vel_mag, bins=30, color='orange', edgecolor='black')
    ax.set_xlabel("Speed")

    # Plot 6: 旋转角度分布
    ax = axes[1, 2]
    # ax.set_title("Rotation (Degrees)")
    # # 将弧度转为度，并映射到 0-180 (因为椭圆旋转具有对称性)
    # deg = (np.degrees(rotation) % 180).flatten()
    # ax.hist(deg, bins=36, range=(0, 180), color='purple', edgecolor='black')
    # ax.set_xlabel("Angle (0-180)")
    J_mask = J.flatten()[J.flatten() < 20]
    ax.set_title("J Distribution (Pixels)")
    ax.hist(J_mask, bins=50, color='skyblue', edgecolor='black')
    ax.set_xlabel("Scale Value")

    plt.tight_layout()
    plt.show()

# 使用方法：
analyze_model_stats(res.model)